# A/B testing 
Business Question: The company runs an expensive training program for salespeople during the last three months of the year (October–December). Only half of the salespeople are trained. We want to know:<br>
Does assigning customers to trained salespeople increase customer spending enough to justify the cost of the training program?<br>
We randomly assigned customers to trained and untrained sales persons. Customers with `CustomerID % 2 == 1` are assigned to trained sales person (treatment group) and customers with `CustomerID % 2 == 0` are assigned to untrained sales persons(control group).<br>
First, let us read the data, filter customers in October to December 2010 interval, then split them into control and treatment group. 

In [1]:
import os
import pandas as pd
import pandasql as psql
from online_retail.utils.base_funcs import load_data

root_path = os.getcwd()
file_path = os.path.abspath(os.path.join(root_path, '..','..','data/online_retail_II.xlsx'))

data = load_data(file_path=file_path)
data.rename(columns = {'Customer ID':'CustomerID'}, inplace = True)
data.head()


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,CustomerID,Country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom


Since some customers just had return transaction (starting with letter 'C')in that period, and did not buy anything and did not guide by sales persons, I do not consider them at all. So if I remove the transactions starting with 'C', then I am sure that those customers are guided by a sales person. There may be some canceled transactions, which could happen exactly in this period or after this period, that are related to the transactions done in this period,  which will not be considered here.  

In [2]:
query = '''
SELECT *
FROM data
WHERE CustomerID IS NOT NULL AND Invoice NOT LIKE 'C%'
'''
data_clean = psql.sqldf(query, locals())
data_clean.info()
data_clean.isnull().sum()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 805620 entries, 0 to 805619
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype  
---  ------       --------------   -----  
 0   Invoice      805620 non-null  object 
 1   StockCode    805620 non-null  object 
 2   Description  805620 non-null  object 
 3   Quantity     805620 non-null  int64  
 4   InvoiceDate  805620 non-null  object 
 5   Price        805620 non-null  float64
 6   CustomerID   805620 non-null  float64
 7   Country      805620 non-null  object 
dtypes: float64(2), int64(1), object(5)
memory usage: 49.2+ MB


Invoice        0
StockCode      0
Description    0
Quantity       0
InvoiceDate    0
Price          0
CustomerID     0
Country        0
dtype: int64

In [3]:
query = '''
SELECT CustomerID, SUM(Quantity * Price) AS Monetary
FROM data_clean
WHERE InvoiceDate BETWEEN '2010-10-01' AND '2011-01-01'
GROUP BY CustomerID
'''
testing_group = psql.sqldf(query = query, env = locals())
display(testing_group)

,CustomerID,Monetary
0,12347.0,2035.11
1,12348.0,892.80
2,12349.0,1402.62
3,12351.0,300.93
4,12352.0,343.80
...,...,...
2665,18278.0,240.30
2666,18280.0,307.55
2667,18283.0,195.35
2668,18284.0,461.68


We have 2670 customers who spend money in this interval. Now we split the customers into 2 treatment and control groups.

In [ ]:
query = '''
SELECT CustomerID, SUM(Quantity * Price) AS Monetary
FROM data_clean
WHERE InvoiceDate BETWEEN '2010-10-01' AND '2011-01-01' AND (CustomerID % 2) = 1
GROUP BY CustomerID
'''
treatment_group = psql.sqldf(query = query, env = locals())
display(treatment_group)

,CustomerID,Monetary
0,12347.0,2035.11
1,12349.0,1402.62
2,12351.0,300.93
3,12353.0,317.76
4,12357.0,12079.99
...,...,...
1332,18259.0,1693.20
1333,18269.0,337.20
1334,18277.0,732.53
1335,18283.0,195.35


In [10]:
query = '''
SELECT CustomerID, SUM(Quantity * Price) AS Monetary
FROM data_clean
WHERE InvoiceDate BETWEEN '2010-10-01' AND '2011-01-01' AND (CustomerID % 2) = 0
GROUP BY CustomerID
'''
control_group = psql.sqldf(query = query, env = locals())
display(control_group)

,CustomerID,Monetary
0,12348.0,892.80
1,12352.0,343.80
2,12356.0,3562.25
3,12358.0,1021.08
4,12360.0,810.79
...,...,...
1328,18270.0,161.40
1329,18276.0,636.99
1330,18278.0,240.30
1331,18280.0,307.55


## Power analysis
In this section, I will determine the prerequsite parameters I need to run the test. $\alpha$, $\beta$, and $\delta$. Other required parameter is minimum number of samples per group that are need to do the experiment $n = \frac{2\sigma^{2}}{\delta_{abs}^{2}}*(z_{1-\alpha}+z_{1-\beta})^{2}$.$\sigma$ is the standard deviation of monetary before doing the experiment, $\delta_{abs} = \delta * \mu_{base}$ in which $\mu_{base}$ is the baseline spending of the control group. Since before Oct 2010, customers did not have access to trained sales persons we can consider this mean as the baseline spending of the control group. For customers before Oct 2010, the canceled transactions is important in finding monetary for each customer. Since some customers may buy some product before 2009 and return it in 2010, I just considered customers with positive `SUM(Quantity * Price)>0`. Since it is one-sided test, I just considered $z_{1-\alpha}$ not $z_{1-\alpha/2}$. 

In [18]:
from scipy.stats import norm

alpha = 0.05
beta = 0.2
delta = 0.35

query = '''
SELECT CustomerID, SUM(Quantity * Price) AS Monetary
FROM data
WHERE CustomerID IS NOT NULL AND InvoiceDate < '2010-10-01'
GROUP BY CustomerID
HAVING SUM(Quantity * Price)>0
'''
before_testing_group = psql.sqldf(query = query, env = locals())
display(before_testing_group)

mean_base = before_testing_group["Monetary"].mean()
std_base = before_testing_group["Monetary"].std()

delta_abs = delta * mean_base # convert relative lift to absolute
    
z_alpha = norm.ppf(1 - alpha)  # one-sided
z_beta  = norm.ppf(1-beta)  

n = 2 * (std_base ** 2) * (z_alpha + z_beta) ** 2 / (delta_abs ** 2)

,CustomerID,Monetary
0,12348.0,222.16
1,12349.0,1244.37
2,12355.0,488.21
3,12358.0,1697.93
4,12359.0,1918.03
...,...,...
3529,18281.0,120.32
3530,18283.0,446.42
3531,18285.0,427.00
3532,18286.0,1188.43


In [19]:
print(control_group["Monetary"].mean())
print(treatment_group["Monetary"].mean())

1125.303414103526
1191.906118922962
